### 1. Lib imports

In [1]:
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, random_split, Subset
from tqdm import tqdm

### 2. Hiperparameters, consts

In [42]:
DATA_DIR = "train"
TEST_DIR = "test"
BEST_MODEL_PATH = "best_model.pth"
IMG_SIZE = 64
BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 0.001
VAL_SPLIT = 0.2
SEED = 42
PATIENCE = 8

### 3. Set the GPU/CPU device and its seed

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True


Device: cuda
GPU: NVIDIA GeForce GTX 1050


### 4. Calc normalization mean and std

In [ ]:
# print("Run to find mean and std of the dataset. If you have already done it, comment out this part to save time.")

# calc_transform = transforms.Compose([
#     transforms.Resize((IMG_SIZE, IMG_SIZE)),
#     transforms.ToTensor()
# ])

# calc_dataset = torchvision.datasets.ImageFolder(DATA_DIR, transform=calc_transform)
# calc_loader = DataLoader(calc_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# mean = torch.zeros(3)
# std = torch.zeros(3)
# total_images = 0

# for images, _ in tqdm(calc_loader, desc="Calculating mean and std"):
#     batch_samples = images.size(0)
#     images = images.view(batch_samples, images.size(1), -1)
    
#     mean += images.mean(2).sum(0)
#     std += images.std(2).sum(0)
#     total_images += batch_samples

# mean /= total_images
# std /= total_images

# custom_mean = mean.tolist()
# custom_std = std.tolist()
# print(f"Calculated mean: {custom_mean}")
# print(f"Calculated std: {custom_std}")

Run to find mean and std of the dataset. If you have already done it, you can comment out this part to save time.


Skanowanie obrazów: 100%|██████████| 1376/1376 [00:59<00:00, 23.26it/s]

Obliczone mean: [0.5203942656517029, 0.4950304925441742, 0.4381066858768463]
Obliczone std: [0.21126312017440796, 0.210307314991951, 0.21001148223876953]


### 5. Data augmentation

In [23]:
custom_mean = [0.5203942656517029, 0.4950304925441742, 0.4381066858768463]
custom_std = [0.21126312017440796, 0.210307314991951, 0.21001148223876953]


train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomCrop(IMG_SIZE, padding=8),
    transforms.RandomGrayscale(p=0.1),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=custom_mean, std=custom_std),
    transforms.RandomErasing(p=0.2),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=custom_mean, std=custom_std)
])

### 6. Load and split the data set

In [24]:
base_dataset = torchvision.datasets.ImageFolder(DATA_DIR)
num_classes = len(base_dataset.classes)

dataset_size = len(base_dataset)
train_size = int((1 - VAL_SPLIT) * dataset_size)
val_size = dataset_size - train_size

generator = torch.Generator().manual_seed(SEED)
train_subset_raw, val_subset_raw = random_split(base_dataset, [train_size, val_size], generator=generator)

train_dataset_full = torchvision.datasets.ImageFolder(DATA_DIR, transform=train_transform)
val_dataset_full = torchvision.datasets.ImageFolder(DATA_DIR, transform=val_transform)

train_dataset = Subset(train_dataset_full, train_subset_raw.indices)
val_dataset = Subset(val_dataset_full, val_subset_raw.indices)

### 7. DataLoader initilization

In [18]:
num_workers = 4 if os.name != "nt" else 0

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=num_workers, 
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=num_workers, 
    pin_memory=torch.cuda.is_available()
)

### 8. Model structure

In [43]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out

class CNN(nn.Module):
    def __init__(self, num_classes=50):
        super().__init__()

        self.prep = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.layer1 = ResidualBlock(32, 64, stride=2)
        self.layer2 = ResidualBlock(64, 128, stride=2)
        self.layer3 = ResidualBlock(128, 256, stride=2)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.prep(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.classifier(x)
        return x

model = CNN(num_classes=num_classes).to(device)

### 9. Criterion, optimizer, scheduler

In [44]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

### 10. Validation function

In [21]:
def evaluate(model, loader, device, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        loop = tqdm(loader, desc="Validation", leave=False)
        for images, labels in loop:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    return avg_loss, acc

### 11. Training with early stop

In [45]:
best_val_loss = float('inf') 
best_model_weights = copy.deepcopy(model.state_dict())
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}] Train", leave=False)
    
    for images, labels in loop:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        loop.set_postfix(loss=loss.item())

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    val_loss, val_acc = evaluate(model, val_loader, device, criterion)

    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"LR: {current_lr:.6f} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("Saved best model to best_model.pth")
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping - no improvement in validation loss.")
        break

model.load_state_dict(best_model_weights)
print(f"Training finished. Best Validation Loss: {best_val_loss:.4f}")

Epoka [1/50] Train:   0%|          | 0/1101 [00:00<?, ?it/s]

Epoka [1/50] | LR: 0.000999 | Train Loss: 3.1485 | Train Acc: 0.2132 | Val Loss: 3.3702 | Val Acc: 0.2028
Saved best model to best_model.pth


Epoka [2/50] | LR: 0.000996 | Train Loss: 2.7499 | Train Acc: 0.3307 | Val Loss: 2.9399 | Val Acc: 0.2973
Saved best model to best_model.pth


Epoka [3/50] | LR: 0.000991 | Train Loss: 2.5598 | Train Acc: 0.3955 | Val Loss: 2.5006 | Val Acc: 0.4124
Saved best model to best_model.pth


Epoka [4/50] | LR: 0.000984 | Train Loss: 2.4224 | Train Acc: 0.4409 | Val Loss: 2.3459 | Val Acc: 0.4618
Saved best model to best_model.pth


...

Epoka [29/50] | LR: 0.000382 | Train Loss: 1.4496 | Train Acc: 0.7801 | Val Loss: 1.8761 | Val Acc: 0.6425
Saved best model to best_model.pth


Epoka [30/50] | LR: 0.000352 | Train Loss: 1.4287 | Train Acc: 0.7880 | Val Loss: 1.8751 | Val Acc: 0.6385
Saved best model to best_model.pth


Epoka [31/50] | LR: 0.000323 | Train Loss: 1.4067 | Train Acc: 0.7957 | Val Loss: 1.8853 | Val Acc: 0.6397


Epoka [32/50] | LR: 0.000294 | Train Loss: 1.3858 | Train Acc: 0.8028 | Val Loss: 1.8816 | Val Acc: 0.6455


KeyboardInterrupt: 

### 12. Final Evaluation on Validation Set (as a proxy for test accuracy)

In [46]:
LOAD_MODEL_PATH = BEST_MODEL_PATH

model.load_state_dict(torch.load(LOAD_MODEL_PATH, weights_only=True))
model.to(device)

final_val_loss, final_val_acc = evaluate(model, val_loader, device, criterion)


print("Results (Validation Set):")
print(f"Val Loss: {final_val_loss:.4f}")
print(f"Val Accuracy: {final_val_acc:.4f} ({final_val_acc * 100:.2f}%)")

Results (Validation Set):
Val Loss: 1.8751
Val Accuracy: 0.6385 (63.85%)


### 13. Save model predictions on test dataset

In [47]:
import os
from PIL import Image

model.load_state_dict(torch.load(BEST_MODEL_PATH, weights_only=True))
model.eval()

with open("pred.csv", "w") as f, torch.no_grad():
    
    for file_name in sorted(os.listdir(TEST_DIR)):
        if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            
            img_path = os.path.join(TEST_DIR, file_name)
            img = Image.open(img_path).convert('RGB')
            img_tensor = val_transform(img).unsqueeze(0).to(device)

            pred = model(img_tensor).argmax(dim=1).item()

            f.write(f"{file_name},{pred}\n")

print("Saved predictions to pred.csv")

Saved predictions to pred.csv
